# Basetable — 2024 US Presidential Election

In [19]:
import pandas as pd
import numpy as np

# ─── Load Polymarket win probabilities as the base of the basetable ───
polymarket = pd.read_csv("../Data/1_Bronze/Polymarket/polymarket_win_probabilities.csv", parse_dates=["date"])

# Keep only Trump probability — this is the dependent variable
basetable = polymarket[["date", "Trump (%)"]].rename(columns={"Trump (%)": "polymarket_trump_prob"})
basetable["polymarket_trump_prob"] = basetable["polymarket_trump_prob"] / 100  # scale to [0, 1]

basetable = basetable.sort_values("date").reset_index(drop=True)

# Lag-1: yesterday's probability (autoregressive feature)
basetable["polymarket_trump_prob_lag1"] = basetable["polymarket_trump_prob"].shift(1)

print(f"Base: {len(basetable)} rows from {basetable['date'].min().date()} to {basetable['date'].max().date()}")
basetable.head()

Base: 124 rows from 2024-07-05 to 2024-11-05


,date,polymarket_trump_prob,polymarket_trump_prob_lag1
0,2024-07-05,0.605,NaN
1,2024-07-06,0.625,0.605
2,2024-07-07,0.625,0.625
3,2024-07-08,0.605,0.625
4,2024-07-09,0.625,0.605


# Timedimension

In [20]:
ELECTION_DAY = pd.Timestamp("2024-11-05")

# days_to_election: positive = before election, negative = after
basetable["days_to_election"] = (ELECTION_DAY - basetable["date"]).dt.days

# ISO week number
basetable["week_nr"] = basetable["date"].dt.isocalendar().week.astype(int)

# Weekend dummy (Saturday=5, Sunday=6)
basetable["weekend"] = basetable["date"].dt.dayofweek.isin([5, 6]).astype(int)

basetable[["date", "days_to_election", "week_nr", "weekend"]].head(10)

,date,days_to_election,week_nr,weekend
0,2024-07-05,123,27,0
1,2024-07-06,122,27,1
2,2024-07-07,121,27,1
3,2024-07-08,120,28,0
4,2024-07-09,119,28,0
5,2024-07-10,118,28,0
6,2024-07-11,117,28,0
7,2024-07-12,116,28,0
8,2024-07-13,115,28,1
9,2024-07-14,114,28,1


# Bluesky

## General

In [21]:
# ─── Load raw Bluesky posts ───
bsky_raw = pd.read_csv("../Data/2_Silver/Bluesky/bsky_US_2024_posts.csv")
bsky_raw["timestamp"] = pd.to_datetime(bsky_raw["timestamp"], format="ISO8601", utc=True)
bsky_raw["date"] = bsky_raw["timestamp"].dt.normalize().dt.tz_localize(None)

# Filter to basetable date range
bsky_raw = bsky_raw[(bsky_raw["date"] >= basetable["date"].min()) &
                    (bsky_raw["date"] <= basetable["date"].max())]

# Trump mention flag via candidate column
bsky_raw["mentions_trump"] = bsky_raw["candidate"] == "CandidateA"

# ─── Aggregate per day ───
bsky_daily = bsky_raw.groupby("date").agg(
    bsky_post_count       = ("uri",           "count"),
    bsky_trump_post_share = ("mentions_trump", "mean"),
    bsky_reply_ratio      = ("is_reply",       "mean"),
    bsky_repost_ratio     = ("reposts",        lambda x: (x > 0).mean()),
).reset_index()

# Merge into basetable
basetable = basetable.merge(bsky_daily, on="date", how="left")

basetable[["date", "bsky_post_count", "bsky_trump_post_share", "bsky_reply_ratio", "bsky_repost_ratio"]].head(10)

,date,bsky_post_count,bsky_trump_post_share,bsky_reply_ratio,bsky_repost_ratio
0,2024-07-05,36.0,0.361111,0.472222,0.222222
1,2024-07-06,23.0,0.434783,0.130435,0.130435
2,2024-07-07,27.0,0.481481,0.296296,0.148148
3,2024-07-08,22.0,0.590909,0.000000,0.454545
4,2024-07-09,33.0,0.454545,0.181818,0.181818
5,2024-07-10,39.0,0.461538,0.205128,0.179487
6,2024-07-11,62.0,0.354839,0.032258,0.112903
7,2024-07-12,43.0,0.372093,0.139535,0.139535
8,2024-07-13,32.0,0.375000,0.187500,0.250000
9,2024-07-14,100.0,0.600000,0.490000,0.310000


## Sentiment

In [22]:
# ─── Bluesky Sentiment — VADER ───────────────────────────────────────────────
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

# Compound score in [-1, 1] per post
bsky_raw["vader"] = bsky_raw["text"].astype(str).apply(
    lambda t: sia.polarity_scores(t)["compound"]
)

# Per-day mean sentiment per candidate
sent_daily = (
    bsky_raw.groupby(["date", "candidate"])["vader"]
    .mean()
    .unstack(fill_value=np.nan)
    .rename(columns={"CandidateA": "bsky_trump_sentiment_avg",
                     "CandidateB": "bsky_harris_sentiment_avg"})
    .reset_index()
)
for col in ["bsky_trump_sentiment_avg", "bsky_harris_sentiment_avg"]:
    if col not in sent_daily.columns:
        sent_daily[col] = np.nan

basetable = basetable.merge(
    sent_daily[["date", "bsky_trump_sentiment_avg", "bsky_harris_sentiment_avg"]],
    on="date", how="left"
)

# Sentiment velocity: 1st difference of Trump sentiment (direction > level)
basetable["bsky_sentiment_velocity"] = basetable["bsky_trump_sentiment_avg"].diff()

print(f"Sentiment scored: {len(bsky_raw):,} posts")
basetable[["date", "bsky_trump_sentiment_avg", "bsky_harris_sentiment_avg", "bsky_sentiment_velocity"]].dropna().head(8)

Sentiment scored: 27,215 posts


,date,bsky_trump_sentiment_avg,bsky_harris_sentiment_avg,bsky_sentiment_velocity
1,2024-07-06,0.094200,0.050900,0.190900
2,2024-07-07,-0.297446,0.205490,-0.391646
3,2024-07-08,-0.238692,0.032386,0.058754
4,2024-07-09,-0.151707,0.197820,0.086986
5,2024-07-10,0.162794,0.179015,0.314501
6,2024-07-11,-0.077805,0.044718,-0.240599
7,2024-07-12,-0.058800,0.072950,0.019005
8,2024-07-13,-0.349158,-0.047050,-0.290358


## Network

In [23]:
# ─── Bluesky Network features — computed daily with NetworkX ─────────────────
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities
from networkx.algorithms.community.quality import modularity as nx_modularity

# ── Bridge nodes: load pre-computed global centrality labels ──────────────────
centrality = pd.read_csv("../Data/2_Silver/Bluesky/bsky_US_2024_centrality.csv")
bridge_nodes = set(centrality.loc[centrality["is_bridge"], "node"])

# ── Normalize author handles: strip domain suffix (e.g. "user.bsky.social" → "user") ──
# centrality["node"] contains handles only; bsky_raw["author"] has full domain
bsky_raw["author_handle"] = bsky_raw["author"].str.split(".").str[0]

# ── Resolve parent_uri → parent author handle (within our dataset) ──────────
uri_to_handle    = dict(zip(bsky_raw["uri"], bsky_raw["author_handle"]))
uri_to_candidate = dict(zip(bsky_raw["uri"], bsky_raw["candidate"]))
bsky_raw["parent_handle"] = bsky_raw["parent_uri"].map(uri_to_handle)

# ── Each author's predominant candidate affiliation (by handle) ───────────────
author_camp = (
    bsky_raw.groupby("author_handle")["candidate"]
    .agg(lambda s: s.value_counts().idxmax())
)

# ── Day-by-day network metrics ────────────────────────────────────────────────
NET_COLS = ["net_density", "net_trump_hub_degree_avg", "net_community_homophily",
            "net_bridge_post_share", "echo_chamber_velocity", "net_modularity_Q"]

records = []
prev_trump_users = set()

for day, day_df in bsky_raw.groupby("date"):
    row = {"date": day}

    # Reply edges: replier handle → parent handle (both must be known)
    edges = day_df.loc[
        day_df["is_reply"] & day_df["parent_handle"].notna(),
        ["author_handle", "parent_handle"]
    ]

    if len(edges) < 5:
        row.update({c: np.nan for c in NET_COLS})
        records.append(row)
        prev_trump_users = set()
        continue

    G = nx.DiGraph()
    G.add_edges_from(zip(edges["author_handle"], edges["parent_handle"]))

    # 1. Density
    row["net_density"] = nx.density(G)

    # 2. Top-10 Trump hub degree (mean in-degree of top-10 CandidateA nodes)
    trump_nodes = {n for n in G.nodes() if author_camp.get(n, "") == "CandidateA"}
    trump_indeg  = sorted((G.in_degree(n) for n in trump_nodes), reverse=True)[:10]
    row["net_trump_hub_degree_avg"] = np.mean(trump_indeg) if trump_indeg else np.nan

    # 3. Community homophily (dyadicity): fraction of edges within same camp
    same = sum(1 for u, v in G.edges()
               if author_camp.get(u, "") == author_camp.get(v, "") != "")
    row["net_community_homophily"] = same / G.number_of_edges()

    # 4. Bridge post share: share of today's authors (by handle) that are global bridge nodes
    today_handles = set(day_df["author_handle"])
    row["net_bridge_post_share"] = (
        len(today_handles & bridge_nodes) / len(today_handles)
        if today_handles else np.nan
    )

    # 5. Echo chamber velocity: fraction of Trump users new to the community today
    today_trump = set(day_df.loc[day_df["candidate"] == "CandidateA", "author_handle"])
    new_entrants = today_trump - prev_trump_users
    row["echo_chamber_velocity"] = (
        len(new_entrants) / len(today_trump) if today_trump else np.nan
    )
    prev_trump_users = today_trump

    # 6. Modularity Q (undirected, greedy community detection)
    G_und = G.to_undirected()
    try:
        comms = list(greedy_modularity_communities(G_und))
        row["net_modularity_Q"] = nx_modularity(G_und, comms)
    except Exception:
        row["net_modularity_Q"] = np.nan

    records.append(row)

net_df = pd.DataFrame(records)
basetable = basetable.merge(net_df, on="date", how="left")

print(f"Network features computed for {len(net_df)} days")
print(f"Bridge nodes in centrality file: {len(bridge_nodes)}")
basetable[["date"] + NET_COLS].dropna(subset=["net_density"]).head(8)

Network features computed for 123 days
Bridge nodes in centrality file: 47


,date,net_density,net_trump_hub_degree_avg,net_community_homophily,net_bridge_post_share,echo_chamber_velocity,net_modularity_Q
0,2024-07-05,0.070513,1.000000,0.363636,0.000000,1.000000,0.661157
2,2024-07-07,0.333333,1.000000,0.500000,0.062500,1.000000,0.666667
7,2024-07-12,0.069444,1.000000,0.200000,0.000000,1.000000,0.720000
9,2024-07-14,0.052381,1.555556,0.227273,0.000000,1.000000,0.759766
10,2024-07-15,0.054945,1.000000,0.500000,0.000000,0.677419,0.843750
11,2024-07-16,0.083333,1.000000,0.333333,0.000000,0.750000,0.500000
12,2024-07-17,0.075758,0.800000,0.000000,0.000000,0.785714,0.833333
13,2024-07-18,0.036842,1.500000,0.285714,0.015625,0.755556,0.716837


# Reddit 

# Newspaper

In [24]:
# ─── News features ────────────────────────────────────────────────────────────
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

sia_news = SentimentIntensityAnalyzer()

# ── Load data ─────────────────────────────────────────────────────────────────
mc_daily   = pd.read_csv("../Data/1_Bronze/Newspapers/mediacloud_daily.csv",
                          parse_dates=["date"])
mc_stories = pd.read_csv("../Data/1_Bronze/Newspapers/mediacloud_stories.csv",
                          parse_dates=["date"])

# ── 1 & 2: Volume metrics — directly from mediacloud_daily ───────────────────
news_volume = mc_daily[["date", "trump", "harris"]].copy()
news_volume["news_title_trump_count"]   = news_volume["trump"]
news_volume["news_attention_asymmetry"] = news_volume["trump"] - news_volume["harris"]

# ── VADER on all story titles (run once) ─────────────────────────────────────
mc_stories["vader"] = mc_stories["title"].astype(str).apply(
    lambda t: sia_news.polarity_scores(t)["compound"]
)
mc_stories["is_negative"] = mc_stories["vader"] < -0.05   # VADER threshold

# ── 3 & 4: Sentiment avg + negative ratio per candidate per day ───────────────
def sentiment_features(df, query_label, prefix):
    sub = df[df["query"] == query_label].groupby("date").agg(
        sentiment_avg  = ("vader",       "mean"),
        neg_ratio      = ("is_negative", "mean"),
    ).reset_index()
    return sub.rename(columns={
        "sentiment_avg": f"news_{prefix}_sentiment_avg",
        "neg_ratio":     f"news_{prefix}_neg_ratio",
    })

trump_sent  = sentiment_features(mc_stories, "trump",  "trump")
harris_sent = sentiment_features(mc_stories, "harris", "harris")

# ── 5 & 6: Topic shares — keyword matching on deduplicated story titles ───────
ECONOMY_KW    = r"\b(inflation|economy|economic|gdp|unemployment|recession|" \
                r"jobs?|wages?|trade|tariff|tax|debt|deficit|interest rate|" \
                r"federal reserve|fed |cost of living|prices?|spending)\b"
IMMIGRATION_KW = r"\b(immigra|immigrant|border|migrant|asylum|deport|" \
                 r"illegal alien|undocumented|daca|visa|sanctuary|wall)\b"

# Use all stories, deduplicated by url per day (avoid double-counting)
stories_dedup = mc_stories.drop_duplicates(subset=["date", "url"]).copy()
stories_dedup["title_lower"] = stories_dedup["title"].str.lower()

stories_dedup["topic_economy"]    = stories_dedup["title_lower"].str.contains(
    ECONOMY_KW,    regex=True, na=False)
stories_dedup["topic_immigration"] = stories_dedup["title_lower"].str.contains(
    IMMIGRATION_KW, regex=True, na=False)

topic_daily = stories_dedup.groupby("date").agg(
    total          = ("url",               "count"),
    econ_count     = ("topic_economy",     "sum"),
    immig_count    = ("topic_immigration", "sum"),
).reset_index()

topic_daily["topic_economy_share"]    = topic_daily["econ_count"]  / topic_daily["total"]
topic_daily["topic_immigration_share"] = topic_daily["immig_count"] / topic_daily["total"]

# ── Merge all news features into basetable ────────────────────────────────────
for df_feat in [
    news_volume[["date", "news_title_trump_count", "news_attention_asymmetry"]],
    trump_sent,
    harris_sent,
    topic_daily[["date", "topic_economy_share", "topic_immigration_share"]],
]:
    basetable = basetable.merge(df_feat, on="date", how="left")

NEWS_COLS = ["news_title_trump_count", "news_attention_asymmetry",
             "news_trump_sentiment_avg", "news_trump_neg_ratio",
             "news_harris_sentiment_avg", "news_harris_neg_ratio",
             "topic_economy_share", "topic_immigration_share"]

print(f"VADER scored: {len(mc_stories):,} story titles")
print(f"Deduplicated stories for topic model: {len(stories_dedup):,}")
basetable[["date"] + NEWS_COLS].dropna(subset=["news_title_trump_count"]).head(8)

C:\Users\verme_hzys4y0\AppData\Local\Temp\ipykernel_27448\781831421.py:48: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  stories_dedup["topic_economy"]    = stories_dedup["title_lower"].str.contains(
C:\Users\verme_hzys4y0\AppData\Local\Temp\ipykernel_27448\781831421.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  stories_dedup["topic_immigration"] = stories_dedup["title_lower"].str.contains(


VADER scored: 235,191 story titles
Deduplicated stories for topic model: 148,877


,date,news_title_trump_count,news_attention_asymmetry,news_trump_sentiment_avg,news_trump_neg_ratio,news_harris_sentiment_avg,news_harris_neg_ratio,topic_economy_share,topic_immigration_share
0,2024-07-05,851.0,592.0,-0.036589,0.356052,-0.016054,0.305019,0.030702,0.007675
1,2024-07-06,456.0,346.0,-0.074378,0.396930,-0.083504,0.372727,0.010373,0.004149
2,2024-07-07,421.0,230.0,-0.038358,0.349169,-0.027279,0.298429,0.021882,0.015317
3,2024-07-08,921.0,656.0,-0.040297,0.332248,-0.027917,0.305660,0.027721,0.011294
4,2024-07-09,1070.0,777.0,-0.031364,0.330841,-0.003810,0.307167,0.032286,0.008726
5,2024-07-10,1131.0,843.0,-0.029308,0.345712,-0.019649,0.305556,0.020475,0.001638
6,2024-07-11,1201.0,830.0,-0.027581,0.329725,-0.047530,0.326146,0.034646,0.003937
7,2024-07-12,1042.0,647.0,-0.018373,0.305182,-0.062824,0.344304,0.030331,0.021140


# Financials

## Market data (daily)
Weekends and holidays have no trading data → forward-fill from last known trading day.
Daily return computed as percentage change vs previous close.

In [ ]:
# ─── Load market data ───
market = pd.read_csv("../Data/1_Bronze/Financials/Storage/market.csv", parse_dates=["Date"])
market = market.rename(columns={
    "Date":        "date",
    "SP500":       "sp500_close",
    "Oil":         "oil_close",
    "VIX":         "vix_close",
    "TenYearBond": "bond_10y",
    "USDIndex":    "usd_index",
})
market = market.sort_values("date").reset_index(drop=True)

# ─── All features computed on trading-day data (before ffill) ───
# This prevents spurious zero-return weekends from polluting rolling windows / lag shifts.

# S&P 500 — Returns & Momentum
market["sp500_return_1d"]        = market["sp500_close"].pct_change(1)  * 100
market["sp500_return_3d"]        = market["sp500_close"].pct_change(3)  * 100
market["sp500_return_7d"]        = market["sp500_close"].pct_change(7)  * 100
market["sp500_return_14d"]       = market["sp500_close"].pct_change(14) * 100
market["sp500_rolling_mean_7d"]  = market["sp500_return_1d"].rolling(7).mean()
market["sp500_rolling_mean_14d"] = market["sp500_return_1d"].rolling(14).mean()

# S&P 500 — Volatility
market["sp500_vol_7d"]   = market["sp500_return_1d"].rolling(7).std()
market["sp500_vol_14d"]  = market["sp500_return_1d"].rolling(14).std()
market["sp500_vol_30d"]  = market["sp500_return_1d"].rolling(30).std()
market["sp500_range_7d"] = (
    (market["sp500_close"].rolling(7).max() - market["sp500_close"].rolling(7).min())
    / market["sp500_close"].rolling(7).mean()
)

# S&P 500 — Level-Based
market["sp500_dist_from_high_30d"] = (
    (market["sp500_close"] - market["sp500_close"].rolling(30).max())
    / market["sp500_close"].rolling(30).max() * 100
)
market["sp500_dist_from_low_30d"] = (
    (market["sp500_close"] - market["sp500_close"].rolling(30).min())
    / market["sp500_close"].rolling(30).min() * 100
)
market["sp500_zscore_30d"] = (
    (market["sp500_close"] - market["sp500_close"].rolling(30).mean())
    / market["sp500_close"].rolling(30).std()
)
_sma7  = market["sp500_close"].rolling(7).mean()
_sma30 = market["sp500_close"].rolling(30).mean()
market["sp500_sma_cross_7_30"] = np.sign(_sma7 - _sma30)  # +1 bullish, -1 bearish, 0 neutral

# S&P 500 — Rate of Change
market["sp500_return_accel"]  = market["sp500_return_1d"].diff()    # 2nd-order: Δ(return)
market["sp500_vol_change_7d"] = market["sp500_vol_7d"].diff(5)      # week-over-week (5 trading days)

# S&P 500 — Lag Features
market["sp500_return_lag1"] = market["sp500_return_1d"].shift(1)
market["sp500_return_lag2"] = market["sp500_return_1d"].shift(2)
market["sp500_return_lag3"] = market["sp500_return_1d"].shift(3)

# Oil — Returns & Momentum
market["oil_return_1d"]       = market["oil_close"].pct_change(1)  * 100
market["oil_return_7d"]       = market["oil_close"].pct_change(7)  * 100
market["oil_return_14d"]      = market["oil_close"].pct_change(14) * 100
market["oil_rolling_mean_7d"] = market["oil_return_1d"].rolling(7).mean()

# Oil — Volatility
market["oil_vol_7d"]  = market["oil_return_1d"].rolling(7).std()
market["oil_vol_14d"] = market["oil_return_1d"].rolling(14).std()

# Oil — Level-Based
market["oil_dist_from_high_30d"] = (
    (market["oil_close"] - market["oil_close"].rolling(30).max())
    / market["oil_close"].rolling(30).max() * 100
)
market["oil_zscore_30d"] = (
    (market["oil_close"] - market["oil_close"].rolling(30).mean())
    / market["oil_close"].rolling(30).std()
)

# Oil — Lag Features
market["oil_return_lag1"] = market["oil_return_1d"].shift(1)
market["oil_return_lag2"] = market["oil_return_1d"].shift(2)

# ─── Reindex to full daily range and forward-fill weekends/holidays ───
# Features computed above on trading days → ffill propagates last known value to weekends
full_range = pd.DataFrame({"date": pd.date_range(market["date"].min(), market["date"].max())})
market = full_range.merge(market, on="date", how="left").ffill()

# ─── Merge into basetable ───
market_cols = [
    "date",
    "sp500_close",
    "sp500_return_1d", "sp500_return_3d", "sp500_return_7d", "sp500_return_14d",
    "sp500_rolling_mean_7d", "sp500_rolling_mean_14d",
    "sp500_vol_7d", "sp500_vol_14d", "sp500_vol_30d", "sp500_range_7d",
    "sp500_dist_from_high_30d", "sp500_dist_from_low_30d",
    "sp500_zscore_30d", "sp500_sma_cross_7_30",
    "sp500_return_accel", "sp500_vol_change_7d",
    "sp500_return_lag1", "sp500_return_lag2", "sp500_return_lag3",
    "oil_close",
    "oil_return_1d", "oil_return_7d", "oil_return_14d", "oil_rolling_mean_7d",
    "oil_vol_7d", "oil_vol_14d",
    "oil_dist_from_high_30d", "oil_zscore_30d",
    "oil_return_lag1", "oil_return_lag2",
    "vix_close", "bond_10y", "usd_index",
]
basetable = basetable.merge(market[market_cols], on="date", how="left")

,date,sp500_close,sp500_return_1d,vix_close,usd_index
0,2024-07-05,5567.189941,NaN,12.48,104.879997
1,2024-07-06,5567.189941,NaN,12.48,104.879997
2,2024-07-07,5567.189941,NaN,12.48,104.879997
3,2024-07-08,5572.850098,0.101670,12.37,105.010002
4,2024-07-09,5576.979980,0.074107,12.51,105.120003
5,2024-07-10,5633.910156,1.020807,12.85,105.050003
6,2024-07-11,5584.540039,-0.876303,12.92,104.440002
7,2024-07-12,5615.350098,0.551703,12.46,104.089996
8,2024-07-13,5615.350098,0.551703,12.46,104.089996
9,2024-07-14,5615.350098,0.551703,12.46,104.089996


## Macro data (monthly)
Published once a month → forward-fill to each day until the next release.
This correctly reflects what was *known* on each day (no lookahead bias).

In [26]:
# ─── Load macro data ───
macros = pd.read_csv("../Data/1_Bronze/Financials/macros.csv", parse_dates=["DATE"])
macros = macros.rename(columns={"DATE": "date"})
macros = macros.sort_values("date").reset_index(drop=True)

# ─── Forward-fill monthly values to daily ───
# Merge on full daily range, then ffill → each day gets the last known macro value
full_range = pd.DataFrame({"date": pd.date_range(basetable["date"].min(), basetable["date"].max())})
macros_daily = full_range.merge(macros, on="date", how="left").ffill()

# RealGDP is quarterly → many NaN even after ffill at start; ffill handles it
# Drop rows still NaN at the very start (before first macro release)
macros_daily = macros_daily.rename(columns={
    "RealDisposableIncome": "macro_real_income",
    "RealGDP":              "macro_real_gdp",
    "UnemploymentRate":     "macro_unemployment",
    "CPIInflation":         "macro_cpi",
    "ConsumerSentiment":    "macro_consumer_sentiment",
})

# Merge into basetable
basetable = basetable.merge(macros_daily, on="date", how="left")

basetable[["date", "macro_unemployment", "macro_cpi", "macro_consumer_sentiment"]].head(10)

,date,macro_unemployment,macro_cpi,macro_consumer_sentiment
0,2024-07-05,NaN,NaN,NaN
1,2024-07-06,NaN,NaN,NaN
2,2024-07-07,NaN,NaN,NaN
3,2024-07-08,NaN,NaN,NaN
4,2024-07-09,NaN,NaN,NaN
5,2024-07-10,NaN,NaN,NaN
6,2024-07-11,NaN,NaN,NaN
7,2024-07-12,NaN,NaN,NaN
8,2024-07-13,NaN,NaN,NaN
9,2024-07-14,NaN,NaN,NaN


# Financials

In [27]:
print(f"Basetable shape: {basetable.shape}")
print()

sections = {
    "Target": [
        "polymarket_trump_prob",
    ],
    "Autoregressive": [
        "polymarket_trump_prob_lag1",
    ],
    "Time": [
        "days_to_election", "week_nr", "weekend",
    ],
    "Bluesky — Volume": [
        "bsky_post_count", "bsky_trump_post_share",
        "bsky_reply_ratio", "bsky_repost_ratio",
    ],
    "Bluesky — Sentiment": [
        "bsky_trump_sentiment_avg", "bsky_harris_sentiment_avg",
        "bsky_sentiment_velocity",
    ],
    "Bluesky — Network": [
        "net_density", "net_trump_hub_degree_avg", "net_community_homophily",
        "net_bridge_post_share", "echo_chamber_velocity", "net_modularity_Q",
    ],
    "News — Volume": [
        "news_title_trump_count", "news_attention_asymmetry",
    ],
    "News — Sentiment": [
        "news_trump_sentiment_avg", "news_trump_neg_ratio",
        "news_harris_sentiment_avg", "news_harris_neg_ratio",
    ],
    "News — Topics": [
        "topic_economy_share", "topic_immigration_share",
    ],
    "Financials — Market": [
        "sp500_close", "sp500_return_1d", "oil_close",
        "vix_close", "bond_10y", "usd_index",
    ],
    "Financials — Macro": [
        "macro_real_income", "macro_real_gdp", "macro_unemployment",
        "macro_cpi", "macro_consumer_sentiment",
    ],
    "Political Events": [
        "debate_day", "debate_day_lag1", "debate_day_lag2",
        "event_assassination1", "event_assassination1_lag1",
        "event_biden_out", "event_biden_out_lag1",
        "event_harris_nom", "event_harris_nom_lag1",
        "event_assassination2", "event_assassination2_lag1",
    ],
    "Lag features (t-1)": [
        "bsky_trump_post_share_lag1", "bsky_trump_sentiment_avg_lag1", "bsky_post_count_lag1",
        "news_title_trump_count_lag1", "news_attention_asymmetry_lag1", "news_trump_sentiment_avg_lag1",
        "vix_close_lag1", "sp500_return_lag1",
    ],
}

all_expected = [col for cols in sections.values() for col in cols]
missing = [c for c in all_expected if c not in basetable.columns]
extra   = [c for c in basetable.columns if c not in all_expected and c != "date"]

for section, cols in sections.items():
    present = [c for c in cols if c in basetable.columns]
    print(f"  [{len(present):>2}/{len(cols)}] {section}")
    for c in cols:
        flag = "✓" if c in basetable.columns else "✗ MISSING"
        print(f"         {flag}  {c}")
    print()

if missing:
    print(f"MISSING columns ({len(missing)}): {missing}")
if extra:
    print(f"Unlisted columns ({len(extra)}): {extra}")

basetable

Basetable shape: (124, 38)

  [ 1/1] Target
         ✓  polymarket_trump_prob

  [ 1/1] Autoregressive
         ✓  polymarket_trump_prob_lag1

  [ 3/3] Time
         ✓  days_to_election
         ✓  week_nr
         ✓  weekend

  [ 4/4] Bluesky — Volume
         ✓  bsky_post_count
         ✓  bsky_trump_post_share
         ✓  bsky_reply_ratio
         ✓  bsky_repost_ratio

  [ 3/3] Bluesky — Sentiment
         ✓  bsky_trump_sentiment_avg
         ✓  bsky_harris_sentiment_avg
         ✓  bsky_sentiment_velocity

  [ 6/6] Bluesky — Network
         ✓  net_density
         ✓  net_trump_hub_degree_avg
         ✓  net_community_homophily
         ✓  net_bridge_post_share
         ✓  echo_chamber_velocity
         ✓  net_modularity_Q

  [ 2/2] News — Volume
         ✓  news_title_trump_count
         ✓  news_attention_asymmetry

  [ 4/4] News — Sentiment
         ✓  news_trump_sentiment_avg
         ✓  news_trump_neg_ratio
         ✓  news_harris_sentiment_avg
         ✓  news_harris_neg_rati

,date,polymarket_trump_prob,polymarket_trump_prob_lag1,days_to_election,week_nr,weekend,bsky_post_count,bsky_trump_post_share,bsky_reply_ratio,bsky_repost_ratio,...,sp500_return_1d,oil_close,vix_close,bond_10y,usd_index,macro_real_income,macro_real_gdp,macro_unemployment,macro_cpi,macro_consumer_sentiment
0,2024-07-05,0.6050,NaN,123,27,0,36.0,0.361111,0.472222,0.222222,...,NaN,83.160004,12.480000,4.272,104.879997,NaN,NaN,NaN,NaN,NaN
1,2024-07-06,0.6250,0.6050,122,27,1,23.0,0.434783,0.130435,0.130435,...,NaN,83.160004,12.480000,4.272,104.879997,NaN,NaN,NaN,NaN,NaN
2,2024-07-07,0.6250,0.6250,121,27,1,27.0,0.481481,0.296296,0.148148,...,NaN,83.160004,12.480000,4.272,104.879997,NaN,NaN,NaN,NaN,NaN
3,2024-07-08,0.6050,0.6250,120,28,0,22.0,0.590909,0.000000,0.454545,...,0.101670,82.330002,12.370000,4.269,105.010002,NaN,NaN,NaN,NaN,NaN
4,2024-07-09,0.6250,0.6050,119,28,0,33.0,0.454545,0.181818,0.181818,...,0.074107,81.410004,12.510000,4.300,105.120003,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,2024-11-01,0.5825,0.6325,4,44,0,764.0,0.506545,0.297120,0.294503,...,0.409251,69.489998,21.879999,4.361,104.279999,17851.4,23586.542,4.2,316.528,71.8
120,2024-11-02,0.5585,0.5825,3,44,1,668.0,0.477545,0.326347,0.247006,...,NaN,NaN,NaN,NaN,NaN,17851.4,23586.542,4.2,316.528,71.8
121,2024-11-03,0.5635,0.5585,2,44,1,875.0,0.368000,0.342857,0.270857,...,NaN,NaN,NaN,NaN,NaN,17851.4,23586.542,4.2,316.528,71.8
122,2024-11-04,0.5855,0.5635,1,45,0,1172.0,0.344710,0.308020,0.261092,...,NaN,NaN,NaN,NaN,NaN,17851.4,23586.542,4.2,316.528,71.8


# Political Data

In [28]:
# ─── debate_day: presidential debate dates ───
# June 27 = Biden-Trump (CNN) — outside our data range (starts July 5)
# September 10 = Harris-Trump (ABC)
basetable["debate_day"]      = basetable["date"].isin(pd.to_datetime(["2024-09-10"])).astype(int)
basetable["debate_day_lag1"] = basetable["debate_day"].shift(1).fillna(0).astype(int)
basetable["debate_day_lag2"] = basetable["debate_day"].shift(2).fillna(0).astype(int)

# ─── Event dummies: one per event ───
def event_dummy(dates, lag=0):
    flags = basetable["date"].isin(pd.to_datetime(dates)).astype(int)
    return flags.shift(lag).fillna(0).astype(int)

# Assassination attempt 1 — July 13 (Butler, PA)
basetable["event_assassination1"]      = event_dummy(["2024-07-13"])
basetable["event_assassination1_lag1"] = event_dummy(["2024-07-13"], lag=1)

# Biden drops out — July 21
basetable["event_biden_out"]      = event_dummy(["2024-07-21"])
basetable["event_biden_out_lag1"] = event_dummy(["2024-07-21"], lag=1)

# Harris nominated — August 19
basetable["event_harris_nom"]      = event_dummy(["2024-08-19"])
basetable["event_harris_nom_lag1"] = event_dummy(["2024-08-19"], lag=1)

# Assassination attempt 2 — September 15
basetable["event_assassination2"]      = event_dummy(["2024-09-15"])
basetable["event_assassination2_lag1"] = event_dummy(["2024-09-15"], lag=1)

# Preview: only rows where something happened
event_cols = [c for c in basetable.columns if c.startswith("event_") or c.startswith("debate_")]
basetable[["date"] + event_cols].loc[basetable[event_cols].sum(axis=1) > 0]

,date,debate_day,debate_day_lag1,debate_day_lag2,event_assassination1,event_assassination1_lag1,event_biden_out,event_biden_out_lag1,event_harris_nom,event_harris_nom_lag1,event_assassination2,event_assassination2_lag1
8,2024-07-13,0,0,0,1,0,0,0,0,0,0,0
9,2024-07-14,0,0,0,0,1,0,0,0,0,0,0
16,2024-07-21,0,0,0,0,0,1,0,0,0,0,0
17,2024-07-22,0,0,0,0,0,0,1,0,0,0,0
45,2024-08-19,0,0,0,0,0,0,0,1,0,0,0
46,2024-08-20,0,0,0,0,0,0,0,0,1,0,0
67,2024-09-10,1,0,0,0,0,0,0,0,0,0,0
68,2024-09-11,0,1,0,0,0,0,0,0,0,0,0
69,2024-09-12,0,0,1,0,0,0,0,0,0,0,0
72,2024-09-15,0,0,0,0,0,0,0,0,0,1,0


In [29]:
# ─── Lag features (dag t-1) ───────────────────────────────────────────────────
# Waarom lags? Features zijn van dag t, target is einde-dag t prijs.
# Een lag geeft het model ook gisteren's waarde → detecteert momentum/trendbreuk.
#
# Lags worden toegevoegd voor:
#   • Bluesky sentiment/volume  — sentimentmomentum (stijgt/daalt het?)
#   • Nieuws volume/sentiment   — mediadruk opbouwend of afnemend?
#   • Markt (VIX, SP500)        — gisteren's angst/return als context

LAG_COLS = {
    # Bluesky
    "bsky_trump_post_share":    "bsky_trump_post_share_lag1",
    "bsky_trump_sentiment_avg": "bsky_trump_sentiment_avg_lag1",
    "bsky_post_count":          "bsky_post_count_lag1",
    # Nieuws
    "news_title_trump_count":    "news_title_trump_count_lag1",
    "news_attention_asymmetry":  "news_attention_asymmetry_lag1",
    "news_trump_sentiment_avg":  "news_trump_sentiment_avg_lag1",
    # Markt
    "vix_close":        "vix_close_lag1",
    "sp500_return_1d":  "sp500_return_lag1",
}

for src, dst in LAG_COLS.items():
    basetable[dst] = basetable[src].shift(1)

lag_cols = list(LAG_COLS.values())
print(f"{len(lag_cols)} lag-features aangemaakt")
basetable[["date"] + lag_cols].head(5)

8 lag-features aangemaakt


,date,bsky_trump_post_share_lag1,bsky_trump_sentiment_avg_lag1,bsky_post_count_lag1,news_title_trump_count_lag1,news_attention_asymmetry_lag1,news_trump_sentiment_avg_lag1,vix_close_lag1,sp500_return_lag1
0,2024-07-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-07-06,0.361111,-0.096700,36.0,851.0,592.0,-0.036589,12.48,NaN
2,2024-07-07,0.434783,0.094200,23.0,456.0,346.0,-0.074378,12.48,NaN
3,2024-07-08,0.481481,-0.297446,27.0,421.0,230.0,-0.038358,12.48,NaN
4,2024-07-09,0.590909,-0.238692,22.0,921.0,656.0,-0.040297,12.37,0.10167
